In [1]:
# Seasonal Recommender for Distributors (May 2025)

import pandas as pd
from datetime import datetime

# 1. Load Data
prev_df = pd.read_excel(r"D:\OneDrive - Nilons Enterprises Pvt Ltd\Desktop\Anadi\Data\YTD 2024-2025 NC_E2.xlsx")
curr_df = pd.read_excel(r"D:\OneDrive - Nilons Enterprises Pvt Ltd\Desktop\Anadi\Data\SAP_apr1st_may31st_NCE2.xlsx")

# 2. Standardize and Clean
prev_df.columns = prev_df.columns.str.strip()
curr_df.columns = curr_df.columns.str.strip()

prev_df['Invoice Date'] = pd.to_datetime(prev_df['Invoice Date'])
curr_df['Invoice Date'] = pd.to_datetime(curr_df['Invoice Date'])

prev_df['Qty'] = pd.to_numeric(prev_df['Qty'], errors='coerce')
curr_df['Invoice Qty'] = pd.to_numeric(curr_df['Invoice Qty'], errors='coerce')

# 🔒 Convert key columns to string to ensure merge compatibility
for df in [prev_df, curr_df]:
    df['C. No'] = df['C. No'].astype(str).str.strip()
    df['Item Code'] = df['Item Code'].astype(str).str.strip()

# 3. Define Time Windows
analysis_month = pd.to_datetime("2025-05-01")
this_month = analysis_month.to_period("M").strftime('%Y-%m')
last_month = (analysis_month - pd.DateOffset(months=1)).to_period("M").strftime('%Y-%m')
last_year_same_month = (analysis_month - pd.DateOffset(years=1)).to_period("M").strftime('%Y-%m')

# Add 'period' column to both
prev_df['period'] = prev_df['Invoice Date'].dt.to_period("M").astype(str)
curr_df['period'] = curr_df['Invoice Date'].dt.to_period("M").astype(str)

# 4. Aggregate Sales by Distributor and Item
# Previous year same month
last_year_sales = prev_df[prev_df['period'] == last_year_same_month]
last_year_grouped = last_year_sales.groupby(['C. No', 'Item Code'])['Qty'].sum().reset_index()
last_year_grouped.rename(columns={'Qty': 'last_year_qty'}, inplace=True)

# Current year April & May (2 months)
curr_2_months = curr_df[curr_df['period'].isin([last_month, this_month])]
curr_grouped = curr_2_months.groupby(['C. No', 'Item Code'])['Invoice Qty'].sum().reset_index()
curr_grouped.rename(columns={'Invoice Qty': 'recent_2mo_qty'}, inplace=True)

# 5. Merge for comparison
reco_df = pd.merge(last_year_grouped, curr_grouped, on=['C. No', 'Item Code'], how='left')
reco_df['recent_2mo_qty'] = reco_df['recent_2mo_qty'].fillna(0)

# 6. Add Item Name and Distributor Name
item_lookup = pd.concat([
    prev_df[['Item Code', 'Item Name']],
    curr_df[['Item Code', 'Item Name']]
]).drop_duplicates()
reco_df = reco_df.merge(item_lookup, on='Item Code', how='left')

dist_lookup = pd.concat([
    prev_df[['C. No', 'C. Name']],
    curr_df[['C. No', 'C. Name']]
]).drop_duplicates()
reco_df = reco_df.merge(dist_lookup, on='C. No', how='left')

# 7. Recommendation Condition: seasonal drop
reco_df['recommend'] = reco_df['last_year_qty'] > (1.5 * reco_df['recent_2mo_qty'])

# 8. Optional: Add Category, Channel
meta = prev_df[['Item Code', 'CATEGORY', 'Channel']].drop_duplicates()
reco_df = reco_df.merge(meta, on='Item Code', how='left')

# 9. Final Result
recommended = reco_df[reco_df['recommend'] == True].copy()
recommended = recommended[['C. No', 'C. Name', 'Item Code', 'Item Name', 'CATEGORY', 'Channel',
                           'last_year_qty', 'recent_2mo_qty']]
recommended = recommended.sort_values(by=['C. No', 'last_year_qty'], ascending=[True, False])

# 10. Show top recommendations
recommended.head(20)

,C. No,C. Name,Item Code,Item Name,CATEGORY,Channel,last_year_qty,recent_2mo_qty
0,100006,MAHALAXMI BAKERY,1400919.0,15 KG 5-6 MM PAPAYA FRUIT PRESERVED RED POU (K...,TOOTY FRUTI,GT,160.0,0.0
1,100006,MAHALAXMI BAKERY,1400919.0,15Kg*1 5-6MM CANDIED FRUIT RED POU KRCHI,TOOTY FRUTI,GT,160.0,0.0
2,100007,MODEL BAKERS AND CONFECTIONERS,1400845.0,15 KG MODEL PAPAYA FRUIT PRESERVE MIX POU,TOOTY FRUTI,GT,300.0,0.0
3,100007,MODEL BAKERS AND CONFECTIONERS,1400845.0,15Kg*1 MODEL CANDIED FRUIT MIX POU,TOOTY FRUTI,GT,300.0,0.0
4,100013,RAM BAKERY,1400919.0,15 KG 5-6 MM PAPAYA FRUIT PRESERVED RED POU (K...,TOOTY FRUTI,GT,300.0,0.0
5,100013,RAM BAKERY,1400919.0,15Kg*1 5-6MM CANDIED FRUIT RED POU KRCHI,TOOTY FRUTI,GT,300.0,0.0
70,100015,S.M. SALES,1400840.0,15 KG POPULAR PAPAYA FRUIT PRESERVED MIX POU,TOOTY FRUTI,GT,32.0,0.0
71,100015,S.M. SALES,1400840.0,15Kg*1 POP CANDIED FRUIT MIX POU,TOOTY FRUTI,GT,32.0,0.0
66,100015,S.M. SALES,1400838.0,15 KG POPULAR PAPAYA FRUIT PRESERVED RED POU,TOOTY FRUTI,GT,30.0,0.0
67,100015,S.M. SALES,1400838.0,15 KG POPULAR PAPAYA FRUIT PRESERVED RED POU,TOOTY FRUTI,INST,30.0,0.0
